In [ ]:
import pandas as pd
import sqlite3

# SQLiteデータベースを作成
conn = sqlite3.connect("acuna.db")

# CSVファイルを読み込む
reference_df = pd.read_csv("/content/2023_acuna jr._data_reference.csv")
sprint_df = pd.read_csv("/content/sprint_speed_2023.csv")
savant_df = pd.read_csv("/content/savant_data_acuna jr.(2023).csv")

# CSVをSQLのテーブルに変換
reference_df.to_sql("reference", conn, if_exists="replace", index=False)
sprint_df.to_sql("sprint_speed", conn, if_exists="replace", index=False)
savant_df.to_sql("savant", conn, if_exists="replace", index=False)

print("3つのテーブルを作成しました！")

3つのテーブルを作成しました！


In [ ]:
query = """
SELECT
    BA AS "打率（AVG）",
    OBP AS "出塁率（OBP）",
    SLG AS "長打率（SLG）",
    OPS AS "OPS",
    HR AS "本塁打（HR）",
    SB AS "盗塁（SB）",
    BB AS "四球（BB）"
FROM reference;
"""

result = pd.read_sql_query(query, conn)

print(result)

   打率（AVG）  出塁率（OBP）  長打率（SLG）    OPS  本塁打（HR）  盗塁（SB）  四球（BB）
0    0.337     0.416     0.596  1.012       41      73      80


In [ ]:
# 表をきれいに表示するための機能を読み込む
from IPython.display import display, HTML

# SQLで抽出したDataFrame（result）をHTMLの表に変換
html = result.to_html(index=False)

# 表全体のデザインを設定
# border-collapse：枠線を1本にまとめる
# width：表の横幅を100%にする
# text-align：文字を中央揃えにする
# table-layout：各列の幅を均等にする
html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse: collapse; width: 100%; text-align: center; table-layout: fixed;">'
)

# 見出し（列名）のデザイン
# border：枠線を付ける
# padding：セル内に余白を付ける
# text-align：中央揃え
html = html.replace(
    '<th>',
    '<th style="border: 1px solid #000; padding: 8px; text-align: center;">'
)

# データ部分（各セル）のデザイン
# border：枠線を付ける
# padding：セル内に余白を付ける
# text-align：中央揃え
html = html.replace(
    '<td>',
    '<td style="border: 1px solid #000; padding: 8px; text-align: center;">'
)

# 作成した表をColab上に表示
display(HTML(html))

打率（AVG）,出塁率（OBP）,長打率（SLG）,OPS,本塁打（HR）,盗塁（SB）,四球（BB）
0.337,0.416,0.596,1.012,41,73,80


In [ ]:
# アクーニャJr.の年度別OBPを手入力
# ※数値はSavantなどで確認したものを入力
obp_data = {
    "2021": 0.394,
    "2022": 0.351,
    "2023": 0.416,
    "2024": 0.351,
    "2025": 0.417
}

# DataFrameに変換
obp_df = pd.DataFrame(
    list(obp_data.items()),
    columns=["年度", "出塁率（OBP）"]
)

# OBPを「.XXX」の形式で表示
obp_df["出塁率（OBP）"] = obp_df["出塁率（OBP）"].apply(
    lambda x: f".{int(x * 1000):03d}" if isinstance(x, float) else x
)

# 表をきれいに表示するための機能を読み込む
from IPython.display import display, HTML

# DataFrameをHTMLの表に変換
html = obp_df.to_html(index=False)

# 表全体のデザイン
# border-collapse：枠線をまとめる
# width：表の横幅
# text-align：中央揃え
html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse: collapse; width: 100%; text-align: center;">'
)

# 年度の列を狭くする
html = html.replace(
    '<th>年度</th>',
    '<th style="border: 1px solid #000; padding: 8px; text-align: center; width: 30%;">年度</th>'
)

# OBPの列を広くする
html = html.replace(
    '<th>出塁率（OBP）</th>',
    '<th style="border: 1px solid #000; padding: 8px; text-align: center; width: 70%;">出塁率（OBP）</th>'
)

# データ部分のデザイン
html = html.replace(
    '<td>',
    '<td style="border: 1px solid #000; padding: 8px; text-align: center;">'
)

# 表を表示
display(HTML(html))

年度,出塁率（OBP）
2021,.394
2022,.351
2023,.416
2024,.351
2025,.417


In [ ]:
query = """
SELECT
    sb AS "盗塁（SB）",
    cs AS "盗塁死（CS）"
FROM reference
"""

result = pd.read_sql_query(query, conn)

print(result)

   盗塁（SB）  盗塁死（CS）
0      73       14


In [ ]:
result["盗塁成功率"] = (
    result["盗塁（SB）"] /
    (result["盗塁（SB）"] + result["盗塁死（CS）"])
    * 100
)

result["盗塁成功率"] = result["盗塁成功率"].round(1)

print(result)

   盗塁（SB）  盗塁死（CS）  盗塁成功率
0      73       14   83.9


In [ ]:
query = """
SELECT
    sprint_speed AS "Sprint Speed"
FROM sprint_speed
WHERE "last_name, first_name" LIKE '%Acu%'
"""

sprint_result = pd.read_sql_query(query, conn)

print(sprint_result)

   Sprint Speed
0          28.0


In [ ]:
# referenceテーブルからBB・SB・CSを取得
query = """
SELECT
    bb AS "四球（BB）",
    sb AS "盗塁（SB）",
    cs AS "盗塁死（CS）"
FROM reference
"""

result = pd.read_sql_query(query, conn)


# SBとCSから盗塁成功率を計算
result["盗塁成功率"] = (
    result["盗塁（SB）"] /
    (result["盗塁（SB）"] + result["盗塁死（CS）"])
    * 100
)

result["盗塁成功率"] = result["盗塁成功率"].round(1)


# Sprint Speedを取得
query_sprint = """
SELECT
    sprint_speed AS "Sprint Speed"
FROM sprint_speed
WHERE "last_name, first_name" LIKE '%Acu%'
"""

sprint_result = pd.read_sql_query(query_sprint, conn)


# Sprint Speedをresultに追加
result["Sprint Speed"] = sprint_result["Sprint Speed"].iloc[0]

result["Sprint Speed"] = sprint_result["Sprint Speed"].iloc[0]

# 表示用に単位を追加
result["Sprint Speed"] = result["Sprint Speed"].map(lambda x: f"{x:.1f} ft/s")

# CSは計算に使ったので削除
result = result.drop(columns=["盗塁死（CS）"])

# 盗塁成功率に「%」を付ける
result["盗塁成功率"] = result["盗塁成功率"].astype(str) + "%"

# resultをHTMLの表に変換
from IPython.display import display, HTML

html = result.to_html(index=False)


# 表全体のデザインを設定
# border-collapse：枠線を1本にまとめる
# width：表の横幅を100%にする
# text-align：文字を中央揃えにする
# table-layout：各列の幅を均等にする
html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse: collapse; width: 100%; text-align: center; table-layout: fixed;">'
)


# 見出し（列名）のデザイン
html = html.replace(
    '<th>',
    '<th style="border: 1px solid #000; padding: 8px; text-align: center;">'
)


# データ部分（各セル）のデザイン
html = html.replace(
    '<td>',
    '<td style="border: 1px solid #000; padding: 8px; text-align: center;">'
)


# 作成した表をColab上に表示
display(HTML(html))


四球（BB）,盗塁（SB）,盗塁成功率,Sprint Speed
80,73,83.9%,28.0 ft/s


In [ ]:
query = """
SELECT
    launch_speed AS "Exit Velocity",
    hardhit_percent AS "Hard-Hit%",
    barrels_per_bbe_percent AS "Barrel%",
    xslg AS "xSLG"
FROM savant
"""

result = pd.read_sql_query(query, conn)

print(result)

   Exit Velocity  Hard-Hit%    Barrel%   xSLG
0           94.7  55.258467  15.329768  0.668


In [ ]:
# 指標を行に変換
result = result.T.reset_index()

# 指標名を変更
result["index"] = [
    "打球速度（Exit Velocity）",
    "ハードヒット率（Hard-Hit%）",
    "バレル率（Barrel%）",
    "期待長打率（xSLG）"
]

# 列名を変更
result.columns = [
    "指標",
    "アクーニャJr."
]

# MLB平均
result["MLB平均"] = [
    88.6,
    37.1,
    7.6,
    0.407
]

# 表示用に文字列へ変換
result["アクーニャJr."] = [
    f"{x:.1f}%" if i in [1, 2]
    else f"{x:.3f}" if i == 3
    else f"{x:.1f}"
    for i, x in enumerate(result["アクーニャJr."])
]

result["MLB平均"] = [
    f"{x:.1f}%" if i in [1, 2]
    else f"{x:.3f}" if i == 3
    else f"{x:.1f}"
    for i, x in enumerate(result["MLB平均"])
]

# HTML形式に変換
from IPython.display import display, HTML

html = result.to_html(index=False)

# 表のデザイン
html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse: collapse; width: 100%; text-align: center; table-layout: fixed;">'
)

html = html.replace(
    '<th>',
    '<th style="border: 1px solid #000; padding: 8px; text-align: center;">'
)

html = html.replace(
    '<td>',
    '<td style="border: 1px solid #000; padding: 8px; text-align: center;">'
)

# 指標列を少し狭くする
html = html.replace(
    '<th style="border: 1px solid #000; padding: 8px; text-align: center;">指標</th>',
    '<th style="border: 1px solid #000; padding: 8px; text-align: center; width: 30%;">指標</th>'
)

display(HTML(html))

指標,アクーニャJr.,MLB平均
打球速度（Exit Velocity）,94.7,88.6
ハードヒット率（Hard-Hit%）,55.3%,37.1%
バレル率（Barrel%）,15.3%,7.6%
期待長打率（xSLG）,0.668,0.407
